In [2]:
import pandas as pd
import numpy as np
import torch
import evaluate
from datasets import Dataset
from transformers import (
    BertTokenizer, 
    BertForSequenceClassification, 
    TrainingArguments, 
    Trainer, 
    DataCollatorWithPadding
)
from sklearn.metrics import classification_report

In [12]:
big_path = "Data/big_data.csv"
smaller_path = "Data/smaller_data.csv"

df = pd.read_csv(big_path)

# Convert Pandas DataFrame to Hugging Face Dataset
hf_dataset = Dataset.from_pandas(df)

# Split into 80% training and 20% testing
split_datasets = hf_dataset.train_test_split(test_size=0.2, seed=42)

# 2. Tokenization with Batching Setup
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # Truncate long articles to 512 tokens. 
    # Note: We don't pad here. The DataCollator will handle dynamic padding later.
    return tokenizer(examples["text"], truncation=True, max_length=512)

# Apply tokenization to the entire dataset
tokenized_datasets = split_datasets.map(tokenize_function, batched=True)

Map: 100%|██████████| 57652/57652 [00:05<00:00, 9891.83 examples/s] 


In [4]:
# 3. Model Initialization
id2label = {0: "real", 1: "fake"}
label2id = {"real": 0, "fake": 1}

model = BertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

# 4. Define Evaluation Metrics
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

# 5. Training Configuration
# DataCollator automatically pads each batch to the length of the longest sentence in *that specific batch*, saving VRAM.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./bert-fake-news-checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5, # Slightly lower learning rate is standard for classic BERT
    per_device_train_batch_size=16, # Pushing this to 16 to utilize your 12GB VRAM
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True, # Critical for RTX 3060 speed
    load_best_model_at_end=True # Automatically swaps to the best performing epoch at the end
)

# 6. Initialize and Run Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8291.86it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider tr

In [5]:
print("Starting training on GPU:", torch.cuda.get_device_name(0))
trainer.train()


Starting training on GPU: NVIDIA GeForce RTX 3060


Epoch,Training Loss,Validation Loss,Accuracy
1,0.211706,0.226979,0.939667
2,0.049685,0.161018,0.963667
3,0.017339,0.279878,0.949667


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.99it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer

TrainOutput(global_step=2250, training_loss=0.08174225319756402, metrics={'train_runtime': 854.5428, 'train_samples_per_second': 42.128, 'train_steps_per_second': 2.633, 'total_flos': 8133428721036480.0, 'train_loss': 0.08174225319756402, 'epoch': 3.0})

In [10]:
#LOADING MODEL, PRETRAINED
from transformers import AutoModelForSequenceClassification, AutoTokenizer

save_path = "./modernBERT-final"

my_model = AutoModelForSequenceClassification.from_pretrained(save_path)
my_tokenizer = AutoTokenizer.from_pretrained(save_path)

Loading weights: 100%|██████████| 138/138 [00:00<00:00, 5519.87it/s]


In [11]:
test_results = trainer.predict(tokenized_datasets["test"])

# The rest of the code remains exactly the same
predicted_labels = np.argmax(test_results.predictions, axis=-1)
actual_labels = test_results.label_ids

print(classification_report(actual_labels, predicted_labels, target_names=["real", "fake"]))

              precision    recall  f1-score   support

        real       0.99      0.94      0.96      1489
        fake       0.94      0.99      0.96      1511

    accuracy                           0.96      3000
   macro avg       0.96      0.96      0.96      3000
weighted avg       0.96      0.96      0.96      3000



In [9]:
save_path = "./classicBERT-final"

trainer.save_model(save_path)

tokenizer.save_pretrained(save_path)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.73it/s]


('./classicBERT-final\\tokenizer_config.json',
 './classicBERT-final\\tokenizer.json')